# Predictive Maintenance AI — End-to-End ML Workflow

**Project:** Predict whether a machine will fail using the UCI AI4I 2020 Predictive Maintenance Dataset.

This notebook brings together the repository's existing preprocessing utilities, exploratory analysis, three classifiers, held-out evaluation, and Random Forest permutation importance. It is designed to be run from the **repository root**.

**Important:** the notebook contains no pre-filled performance numbers. Run the cells to calculate metrics from your local dataset; do not copy or report results until you have executed them.

## 1. Objectives and workflow

1. Load and validate the dataset.
2. Inspect data quality, target imbalance, feature distributions, and correlations.
3. Clean exact duplicates and invalid targets while reporting (not automatically removing) outliers.
4. Exclude identifiers and failure-mode flags that leak the target.
5. Create one reproducible stratified 80/20 split.
6. Fit Logistic Regression, Decision Tree, and Random Forest pipelines with training-only preprocessing.
7. Compare accuracy, precision, recall, F1, ROC-AUC, and confusion matrices.
8. Inspect Random Forest permutation importance and document limitations.

**Evaluation note:** failures are a minority class. Accuracy by itself can look strong even when a model misses failures. We therefore inspect failure-class recall, F1, and false negatives as well.

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    RocCurveDisplay,
    roc_auc_score,
)
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from src.preprocessing import (
    DEFAULT_DATA_PATH,
    IDENTIFIER_COLUMNS,
    LEAKAGE_COLUMNS,
    TARGET_COLUMN,
    clean_dataset,
    load_dataset,
    prepare_data,
)

RANDOM_STATE = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

print(f"Dataset path: {DEFAULT_DATA_PATH}")
print(f"Dataset exists: {DEFAULT_DATA_PATH.exists()}")

## 2. Load the dataset

Download the CSV from the [official UCI AI4I 2020 dataset page](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset) and save it as `data/ai4i2020.csv`. The raw dataset is intentionally excluded from Git.

In [ ]:
raw_df = load_dataset()
print(f"Raw shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
display(raw_df.head())
display(raw_df.dtypes.rename("dtype").to_frame())
display(raw_df.describe(include="all").T)

## 3. Data quality and cleaning

The shared project utility removes exact duplicate rows and rows with a missing target. Missing feature values are retained for imputation inside the model pipeline. IQR outliers are reported, not automatically deleted, because unusual sensor measurements may be meaningful.

In [ ]:
df, cleaning_report = clean_dataset(raw_df)

print("Cleaning summary")
print("----------------")
for key, value in cleaning_report.items():
    print(f"{key}: {value}")

print("\nMissing values by column")
display(df.isna().sum().rename("missing_count").to_frame())

## 4. Exploratory data analysis

### 4.1 Target distribution

The positive class is `Machine failure = 1`. Check the actual class counts and proportions in this run before interpreting any model metrics.

In [ ]:
target_counts = df[TARGET_COLUMN].value_counts().sort_index()
target_share = df[TARGET_COLUMN].value_counts(normalize=True).sort_index()

display(pd.DataFrame({
    "count": target_counts,
    "proportion": target_share,
}).rename(index={0: "No failure (0)", 1: "Failure (1)"}))

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x=TARGET_COLUMN, ax=ax)
ax.set(title="Machine failure class distribution", xlabel="Machine failure (0 = no, 1 = yes)", ylabel="Rows")
plt.tight_layout()
plt.show()

### 4.2 Numeric feature distributions

These plots help reveal ranges and skew. They are exploratory, not evidence that any feature causes failure.

In [ ]:
numeric_features = [
    c for c in df.select_dtypes(include="number").columns
    if c not in [TARGET_COLUMN, *IDENTIFIER_COLUMNS, *LEAKAGE_COLUMNS]
]
display(df[numeric_features].describe().T)

df[numeric_features].hist(figsize=(12, 8), bins=30, edgecolor="black")
plt.suptitle("Numeric sensor-feature distributions")
plt.tight_layout()
plt.show()

### 4.3 Correlations

Correlations describe pairwise linear association in this dataset; they do not establish causation. Identifier columns and failure-mode flags are excluded.

In [ ]:
corr = df[numeric_features].corr()
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", ax=ax)
ax.set_title("Numeric sensor-feature correlations")
plt.tight_layout()
plt.show()

## 5. Feature selection and leakage prevention

- **Target:** `Machine failure`.
- **Excluded identifiers:** `UDI` and `Product ID`.
- **Excluded leakage flags:** `TWF`, `HDF`, `PWF`, `OSF`, and `RNF`. These indicate failure modes and would expose information closely tied to the target.
- **Retained category:** `Type`.
- Numeric features use median imputation and standard scaling.
- Categorical features use most-frequent imputation and one-hot encoding.

All transformations are part of each model's pipeline, so the imputers, scaler, and encoder are fitted only on training rows.

In [ ]:
data = prepare_data(df, test_size=0.20, random_state=RANDOM_STATE)
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Feature count before encoding: {X_train.shape[1]}")
print("Features:", list(X_train.columns))
print("\nTraining target proportions:")
display(y_train.value_counts(normalize=True).sort_index().rename("proportion").to_frame())
print("Test target proportions:")
display(y_test.value_counts(normalize=True).sort_index().rename("proportion").to_frame())

assert not set(LEAKAGE_COLUMNS).intersection(X_train.columns), "Failure-mode leakage columns found!"
assert not set(IDENTIFIER_COLUMNS).intersection(X_train.columns), "Identifier columns found!"
assert len(X_train) + len(X_test) == len(df)
print("Leakage checks passed.")

## 6. Train and evaluate three models

The same split and shared preprocessing design are used for all three models. Class weighting is used to account for imbalance, but it does not guarantee that the model will achieve high failure recall.

The held-out test set is used for final comparison only. If you later tune parameters or choose a decision threshold, do so using a separate validation set or cross-validation rather than repeatedly optimizing against this test set.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE,
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=8,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

fitted_models = {}
metric_rows = []
confusion_matrices = {}
classification_reports = {}

for name, estimator in models.items():
    pipeline = Pipeline([
        ("preprocessor", clone(data["preprocessor"])),
        ("model", clone(estimator)),
    ])
    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, predictions, labels=[0, 1])

    fitted_models[name] = pipeline
    confusion_matrices[name] = cm
    classification_reports[name] = classification_report(
        y_test, predictions, labels=[0, 1], target_names=["No failure", "Failure"],
        zero_division=0, output_dict=True
    )
    metric_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
        "true_negatives": int(cm[0, 0]),
        "false_positives": int(cm[0, 1]),
        "false_negatives": int(cm[1, 0]),
        "true_positives": int(cm[1, 1]),
    })

comparison = pd.DataFrame(metric_rows).sort_values("f1", ascending=False).reset_index(drop=True)
display(comparison.style.format({
    "accuracy": "{:.3f}", "precision": "{:.3f}", "recall": "{:.3f}",
    "f1": "{:.3f}", "roc_auc": "{:.3f}"
}))
comparison.to_csv(OUTPUT_DIR / "notebook_model_comparison.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'notebook_model_comparison.csv'}")

### 6.1 Confusion matrices

For the positive class (failure):
- **False negative (FN):** a failure was present but the model predicted no failure.
- **False positive (FP):** the model raised an alarm when the observed label was no failure.

A false negative can be costly in maintenance settings, while false positives can lead to unnecessary inspections. The appropriate trade-off depends on real operational costs, which this dataset does not provide.

In [ ]:
fig, axes = plt.subplots(1, len(fitted_models), figsize=(15, 4))
for ax, (name, pipeline) in zip(axes, fitted_models.items()):
    ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrices[name],
        display_labels=["No failure", "Failure"],
    ).plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(name)
plt.tight_layout()
plt.show()

### 6.2 ROC curves

ROC-AUC measures ranking across thresholds, whereas precision, recall, and F1 above use each classifier's default prediction threshold. With an imbalanced target, also consider precision-recall curves if further threshold analysis is required.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, pipeline in fitted_models.items():
    RocCurveDisplay.from_estimator(pipeline, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
ax.set_title("ROC curves — held-out test set")
plt.tight_layout()
plt.show()

### 6.3 Classification reports

Review the failure-class precision and recall for each model rather than relying only on the overall accuracy.

In [ ]:
for name, report in classification_reports.items():
    print(f"\n{name}")
    print("-" * len(name))
    display(pd.DataFrame(report).T.round(3))

## 7. Random Forest permutation importance

Permutation importance estimates how much the selected evaluation score decreases when one original input feature is shuffled. This notebook uses held-out F1 and evaluates the full fitted pipeline on raw input columns. Importance is model- and dataset-dependent and does **not** prove causation.

In [ ]:
rf_pipeline = fitted_models["Random Forest"]
importance = permutation_importance(
    rf_pipeline,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

display(importance_df.round(4))
importance_df.to_csv(OUTPUT_DIR / "notebook_random_forest_permutation_importance.csv", index=False)

top = importance_df.head(10).sort_values("importance_mean")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"])
ax.set_xlabel("Decrease in held-out F1 after permutation")
ax.set_title("Random Forest permutation importance")
plt.tight_layout()
plt.show()
print(f"Saved: {OUTPUT_DIR / 'notebook_random_forest_permutation_importance.csv'}")

## 8. Results interpretation and model selection

Use the actual comparison table and confusion matrices generated above to complete this section after running the notebook.

Answer these questions in your own words:

1. Which model has the strongest failure-class recall? How many failures does it miss?
2. Which model has the best F1-score? Does it also have acceptable precision?
3. What is the trade-off between false negatives and false positives for each model?
4. Does the ROC-AUC ranking match the F1 ranking? Why might they differ?
5. Which model would you select for a **prototype**, given the trade-off? What additional operational information would be needed before deployment?

**Do not select a winner based on model complexity or accuracy alone.** A real maintenance decision should account for the cost of missed failures, inspection costs, lead time, calibration, and the performance of the system on real machines.

In [ ]:
# A small data-driven summary to help write the interpretation.
# This prints measured values from the current run; it does not make the business decision for you.
best_f1_row = comparison.loc[comparison["f1"].idxmax()]
best_recall_row = comparison.loc[comparison["recall"].idxmax()]

print("Highest F1 in this run:")
display(best_f1_row.to_frame("value"))
print("\nHighest failure recall in this run:")
display(best_recall_row.to_frame("value"))

print("\nFalse-negative counts by model:")
display(comparison[["model", "false_negatives", "false_positives", "recall", "precision", "f1"]])

## 9. Limitations and future work

- **Dataset realism:** AI4I 2020 is a synthetic dataset designed to resemble industrial predictive-maintenance data; results may not transfer to real machines.
- **Class imbalance:** failure examples are relatively uncommon. Metrics can change with the test split and decision threshold.
- **Single hold-out split:** the 80/20 split provides a reproducible comparison, but repeated validation or cross-validation would give a more robust estimate.
- **Threshold choice:** default thresholds may not match the cost of missed failures versus unnecessary inspections.
- **No operational cost data:** this dataset does not quantify downtime, maintenance cost, or lead time, so business value cannot be claimed from classification metrics alone.
- **Permutation importance:** indicates predictive reliance in this evaluation, not causal relationships.
- **No deployment validation:** before real use, evaluate on data from actual machines, monitor drift, and validate model calibration and safety processes.

Potential next steps: stratified cross-validation, validation-only threshold selection, calibration, saving a tested pipeline, and a small Streamlit demo.

## 10. Reproducibility and saved outputs

This notebook uses random seed **42**, a stratified **80/20** split, and the repository's shared preprocessing module. The metrics and permutation importance CSVs are written to `outputs/`, which is excluded from Git by default. Regenerate them by rerunning the notebook.

Before submission:

- Run **Kernel → Restart Kernel and Run All Cells**.
- Confirm every cell completes without errors.
- Keep the executed notebook with its actual outputs for the submission, but do not commit the raw dataset.
- Make sure the report and slides use values from the same verified run.
- Explain the leakage exclusions, class imbalance, false negatives, model limitations, and future scope during the viva.